MedMCQA: GPT-OSS-120B baseline..


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/external_benchmarks/medmcqa/benchmark_subsets'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import os
import json
import hashlib
import re
import time
from typing import Dict, Any, List
import pandas as pd
import requests
from openai import OpenAI
from tqdm.auto import tqdm
from pathlib import Path
import getpass


In [ ]:
FOLDER_ID = ''
API_KEY = ''
LLM_API_URL = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'
BASELINE_MODEL_NAME = 'gpt_default'
BASELINE_MODEL_URI = f'gpt://{FOLDER_ID}/gpt-oss-120b/latest'
CASE_COMPONENT_COLUMNS = ['question', 'option_a', 'option_b', 'option_c', 'option_d']
COLUMN_DISPLAY_NAMES: Dict[str, str] = {'question': 'Question', 'option_a': 'Option A', 'option_b': 'Option B', 'option_c': 'Option C', 'option_d': 'Option D'}
SAMPLE_SIZE = None
RANDOM_SEED = 42
REQUEST_TIMEOUT = 120
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
TEMPERATURE_PROTOCOL = 0.3
MAX_TOKENS_PROTOCOL = 3000
BASELINE_ROLE_NAME = 'Default'
BASELINE_ROLE_DESCRIPTION = 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'
ANSWER_FIELD_NAME = 'selected_option'
ANSWER_LABELS = ['A', 'B', 'C', 'D']

def detect_provider(model_uri: str) -> str:
    if 'aliceai-llm' in model_uri:
        return 'yandex_rest'
    return 'openai_compatible'
BASELINE_PROVIDER = detect_provider(BASELINE_MODEL_URI)
MODEL_MAX_OUTPUT_TOKENS = {'gpt-oss-120b': MAX_TOKENS_PROTOCOL}


In [ ]:
def detect_provider(model_uri: str) -> str:
    if 'aliceai-llm' in model_uri:
        return 'yandex_rest'
    return 'openai_compatible'

def build_case_text(row_data: Dict[str, Any]) -> str:
    parts: List[str] = []
    for column in CASE_COMPONENT_COLUMNS:
        value = row_data.get(column)
        if pd.isna(value):
            continue
        value_str = str(value).strip()
        if not value_str or value_str.lower() == 'nan':
            continue
        label = COLUMN_DISPLAY_NAMES.get(column, column)
        parts.append(f'{label}:\n{value_str}')
    return '\n\n'.join(parts)


In [ ]:
from pathlib import Path
import getpass
import sqlite3
import threading
import math
from collections.abc import MutableMapping
from datetime import datetime, timezone
PACKAGE_NAME = 'shared_benchmark_runs'

def locate_notebook_directory():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        candidates = [parent, parent / 'benchmark_subsets', parent / 'experiments' / 'external_benchmarks/medmcqa' / 'benchmark_subsets', parent / 'experiments' / 'external_benchmarks/medmcqa']
        for candidate in candidates:
            if (candidate / PACKAGE_NAME / 'data_manifest.json').is_file():
                return candidate
    raise FileNotFoundError('Open this notebook within the project containing ' + PACKAGE_NAME)
NOTEBOOK_DIR = locate_notebook_directory()
PACKAGE_DIR = NOTEBOOK_DIR / PACKAGE_NAME
MANIFEST_PATH = PACKAGE_DIR / 'data_manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
SUBSET_ORDER = ['random_dev300', 'shao_hard100', 'goa100', 'marc100', 'marc250', 'dev4183']
assert MANIFEST['subset_order'] == SUBSET_ORDER
SUBSETS = {item['id']: item for item in MANIFEST['subsets']}
RUN_TAG = 'primary'
SKIP_COMPLETED = True
CHECKPOINT_BATCH_SIZE = 25
BUDGET_LIMIT_RUB = 98000.0
BUDGET_SAFETY_BUFFER_RUB = 1500.0
PRICES_RUB_PER_1000 = {'gpt-oss-120b': {'input': 0.3, 'output': 0.3}, 'qwen3.6-35b-a3b': {'input': 0.2, 'output': 0.3}, 'deepseek-v4-flash': {'input': 0.3, 'output': 0.5}}
PRICE_SOURCE = 'https://aistudio.yandex.ru/ru/docs/ai-studio/pricing'
RUNTIME_DIR = PACKAGE_DIR / 'runs' / RUN_TAG
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
LEDGER_PATH = RUNTIME_DIR / 'usage.sqlite3'

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def load_subset(subset_id):
    specification = SUBSETS[subset_id]
    path = PACKAGE_DIR / specification['csv']
    if sha256_file(path) != specification['sha256']:
        raise ValueError(f'Input checksum differs from the pinned manifest: {subset_id}')
    frame = pd.read_csv(path, sep=';', encoding='utf-8-sig', keep_default_na=False)
    if len(frame) != specification['n'] or frame['case_id'].duplicated().any():
        raise ValueError(f'Unexpected size or duplicate IDs: {subset_id}')
    required = ['case_id', *CASE_COMPONENT_COLUMNS, 'correct_option_letter']
    if any((column not in frame.columns for column in required)):
        raise ValueError(f'Missing required input columns: {subset_id}')
    if not frame['correct_option_letter'].isin(ANSWER_LABELS).all():
        raise ValueError(f'Invalid reference answer labels: {subset_id}')
    if frame[CASE_COMPONENT_COLUMNS].apply(lambda column: column.astype(str).str.strip().eq('')).any().any():
        raise ValueError(f'Empty question or answer option: {subset_id}')
    frame['case_text'] = frame.apply(lambda row: build_case_text(row.to_dict()), axis=1)
    return frame
print('Run sequence:', [(name, SUBSETS[name]['n']) for name in SUBSET_ORDER])
print('Total evaluations per system:', sum((SUBSETS[name]['n'] for name in SUBSET_ORDER)))
print('MARC origins:', {name: SUBSETS[name]['split_counts'] for name in ['marc100', 'marc250']})


In [ ]:
session = requests.Session()
llm_cache: Dict[str, Dict[str, Any]] = {}
openai_client = OpenAI(api_key=API_KEY, base_url='https://llm.api.cloud.yandex.net/v1', project=FOLDER_ID)

def make_cache_key(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int) -> str:
    payload = {'provider': provider, 'model_uri': model_uri, 'system_text': system_text, 'user_text': user_text, 'temperature': temperature, 'max_tokens': max_tokens}
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _build_token_dict(prompt_tokens: int=0, completion_tokens: int=0, total_tokens: int=0) -> Dict[str, int]:
    return {'prompt_tokens': int(prompt_tokens or 0), 'completion_tokens': int(completion_tokens or 0), 'total_tokens': int(total_tokens or 0)}

def _extract_openai_text(response: Any) -> str:
    if not getattr(response, 'choices', None):
        return ''
    message = response.choices[0].message
    content = getattr(message, 'content', '')
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                parts.append(item.get('text', ''))
            elif hasattr(item, 'type') and getattr(item, 'type', None) == 'text':
                parts.append(getattr(item, 'text', ''))
        return '\n'.join([p for p in parts if p]).strip()
    return ''

def _extract_openai_usage(usage: Any) -> Dict[str, int]:
    if usage is None:
        return _build_token_dict()
    prompt_tokens = getattr(usage, 'prompt_tokens', None)
    completion_tokens = getattr(usage, 'completion_tokens', None)
    total_tokens = getattr(usage, 'total_tokens', None)
    if total_tokens is None:
        input_tokens = getattr(usage, 'input_tokens', None)
        output_tokens = getattr(usage, 'output_tokens', None)
        prompt_tokens = prompt_tokens if prompt_tokens is not None else input_tokens
        completion_tokens = completion_tokens if completion_tokens is not None else output_tokens
        total_tokens = int(prompt_tokens or 0) + int(completion_tokens or 0)
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def _extract_yandex_usage(result: Dict[str, Any]) -> Dict[str, int]:
    usage = result.get('usage', {}) or {}
    prompt_tokens = usage.get('inputTextTokens', 0) or 0
    completion_tokens = usage.get('completionTokens', 0) or 0
    total_tokens = usage.get('totalTokens', 0) or prompt_tokens + completion_tokens
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def call_yandex_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, timeout: int=REQUEST_TIMEOUT, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    cache_key = make_cache_key('yandex_rest', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        return {'text': llm_cache[cache_key]['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    headers = {'Content-Type': 'application/json', 'Authorization': f'Api-Key {API_KEY}'}
    payload = {'modelUri': model_uri, 'completionOptions': {'stream': False, 'temperature': temperature, 'maxTokens': max_tokens}, 'messages': [{'role': 'system', 'text': system_text}, {'role': 'user', 'text': user_text}]}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = session.post(LLM_API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.RequestException as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        if response.status_code == 200:
            data = response.json()
            result = data.get('result', {}) or {}
            raw_text = result.get('alternatives', [{}])[0].get('message', {}).get('text', '').strip()
            usage = _extract_yandex_usage(result)
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, **usage, 'cached': False}
        last_error_text = f'HTTP {response.status_code}: {response.text}'
        if response.status_code in (429, 500, 502, 503, 504):
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        raise RuntimeError(last_error_text)
    raise RuntimeError(f'Yandex REST call failed after {retries} retries. Last error: {last_error_text}')

def call_openai_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    cache_key = make_cache_key('openai_compatible', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        return {'text': llm_cache[cache_key]['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = openai_client.chat.completions.create(model=model_uri, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=temperature, max_tokens=max_tokens)
            raw_text = _extract_openai_text(response)
            usage = _extract_openai_usage(getattr(response, 'usage', None))
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, **usage, 'cached': False}
        except Exception as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
    raise RuntimeError(f'OpenAI-compatible call failed after {retries} retries. Last error: {last_error_text}')

def call_llm(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int) -> Dict[str, Any]:
    if provider == 'yandex_rest':
        return call_yandex_llm(model_uri, system_text, user_text, temperature, max_tokens)
    if provider == 'openai_compatible':
        return call_openai_llm(model_uri, system_text, user_text, temperature, max_tokens)
    raise ValueError(f'Unknown provider: {provider}')


In [ ]:
def build_stage1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are an expert medical exam reviewer.\nAt this stage, you need to evaluate a single-best-answer medical multiple-choice exam question (four options, exactly one correct) and provide a final answer based on it.\n\nUse only information explicitly present in the exam question. Do not invent facts beyond it.\nIf a detail is absent, write "Not specified"; if a field is unnecessary, write "None". Be careful and\nprecise.'.strip()

def build_stage1_user_prompt(case_text: str, author_id: str) -> str:
    answer_list = json.dumps(ANSWER_LABELS)
    return f'Read the exam question and write one answer.\n\nExam question:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nWrite the protocol in plain text. Same field labels and order:\n\nSelected option:\nRationale:\nConfidence:\nProtocol Author: {author_id}\n\nField instructions:\nSelected option: one of {answer_list}. No additions, no other text on this line.\nRationale: 2-5 sentences explaining your reasoning, grounded only in the exam question content.\nConfidence: a single integer 0-100.\n'.strip()

def strip_author_section(protocol_text: str) -> str:
    if protocol_text is None:
        return ''
    text = str(protocol_text).strip()
    parts = re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)
    return parts[0].rstrip()

def _normalize_label(value: Any, allowed: List[str]) -> str:
    if value is None:
        return ''
    s = str(value).strip()
    if not s:
        return ''
    lookup = {a.lower(): a for a in allowed}
    return lookup.get(s.lower(), '')

def extract_field_value(text: str, field: str) -> str:
    if text is None:
        return ''
    m = re.search(f'{re.escape(field)}\\s*:\\s*(.+)', str(text), re.IGNORECASE)
    return m.group(1).strip() if m else ''

def extract_final_answer(protocol_text: str) -> str:
    return _normalize_label(extract_field_value(protocol_text, 'Selected option'), ANSWER_LABELS)


In [ ]:
def run_baseline_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    system_text = build_stage1_system_prompt(BASELINE_ROLE_NAME, BASELINE_ROLE_DESCRIPTION)
    for _, row in tqdm(input_df.iterrows(), total=len(input_df), desc='Baseline pipeline'):
        row_data = row.to_dict()
        case_text = row_data['case_text']
        user_text = build_stage1_user_prompt(case_text, BASELINE_MODEL_NAME)
        response = call_llm(provider=BASELINE_PROVIDER, model_uri=BASELINE_MODEL_URI, system_text=system_text, user_text=user_text, temperature=TEMPERATURE_PROTOCOL, max_tokens=MAX_TOKENS_PROTOCOL)
        full_protocol = response['text'].strip()
        row_data['baseline_writeup'] = full_protocol
        row_data['baseline_writeup_anonymous'] = strip_author_section(full_protocol)
        row_data['baseline_selected_option'] = extract_final_answer(full_protocol)
        row_data['baseline_model_id'] = BASELINE_MODEL_NAME
        row_data['baseline_model_uri'] = BASELINE_MODEL_URI
        row_data['baseline_provider'] = BASELINE_PROVIDER
        row_data['baseline_role_name'] = BASELINE_ROLE_NAME
        row_data['baseline_role_description'] = BASELINE_ROLE_DESCRIPTION
        row_data['prompt_tokens'] = response['prompt_tokens']
        row_data['completion_tokens'] = response['completion_tokens']
        row_data['total_tokens'] = response['total_tokens']
        row_data['cached'] = response['cached']
        rows.append(row_data)
    return pd.DataFrame(rows)


In [ ]:
def build_token_summary(result_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([{'model_id': BASELINE_MODEL_NAME, 'model_uri': BASELINE_MODEL_URI, 'provider': BASELINE_PROVIDER, 'role_name': BASELINE_ROLE_NAME, 'prompt_tokens': int(result_df['prompt_tokens'].fillna(0).sum()), 'completion_tokens': int(result_df['completion_tokens'].fillna(0).sum()), 'total_tokens': int(result_df['total_tokens'].fillna(0).sum()), 'n_cases': len(result_df), 'n_cached': int(result_df['cached'].fillna(False).sum())}])


In [ ]:
_request_details = threading.local()

class PersistentSubsetCache(MutableMapping):

    def __init__(self, path, namespace):
        self.namespace = namespace
        self.lock = threading.RLock()
        self.connection = sqlite3.connect(path, timeout=60, check_same_thread=False)
        self.connection.execute('PRAGMA journal_mode=WAL')
        self.connection.executescript('\n            CREATE TABLE IF NOT EXISTS responses (\n                namespace TEXT NOT NULL, cache_key TEXT NOT NULL, payload TEXT NOT NULL,\n                PRIMARY KEY(namespace, cache_key));\n            CREATE TABLE IF NOT EXISTS usage_events (\n                event_id INTEGER PRIMARY KEY AUTOINCREMENT,\n                namespace TEXT NOT NULL, cache_key TEXT NOT NULL, model_uri TEXT NOT NULL,\n                prompt_tokens INTEGER NOT NULL, completion_tokens INTEGER NOT NULL,\n                total_tokens INTEGER NOT NULL, reasoning_tokens INTEGER,\n                finish_reason TEXT, max_tokens INTEGER, cost_rub REAL NOT NULL,\n                timestamp_utc TEXT NOT NULL);\n        ')
        self.connection.commit()

    def __getitem__(self, key):
        with self.lock:
            row = self.connection.execute('SELECT payload FROM responses WHERE namespace=? AND cache_key=?', (self.namespace, key)).fetchone()
        if row is None:
            raise KeyError(key)
        return json.loads(row[0])

    def __setitem__(self, key, value):
        details = getattr(_request_details, 'latest', None)
        with self.lock:
            self.connection.execute('INSERT OR REPLACE INTO responses VALUES (?, ?, ?)', (self.namespace, key, json.dumps(value, ensure_ascii=False)))
            if details is not None:
                self._insert_usage(key, details)
                _request_details.latest = None
            self.connection.commit()

    def __delitem__(self, key):
        with self.lock:
            cursor = self.connection.execute('DELETE FROM responses WHERE namespace=? AND cache_key=?', (self.namespace, key))
            self.connection.commit()
        if not cursor.rowcount:
            raise KeyError(key)

    def __iter__(self):
        with self.lock:
            keys = [row[0] for row in self.connection.execute('SELECT cache_key FROM responses WHERE namespace=?', (self.namespace,))]
        return iter(keys)

    def __len__(self):
        with self.lock:
            return self.connection.execute('SELECT COUNT(*) FROM responses WHERE namespace=?', (self.namespace,)).fetchone()[0]

    def clear(self):
        with self.lock:
            self.connection.execute('DELETE FROM responses WHERE namespace=?', (self.namespace,))
            self.connection.commit()

    def _insert_usage(self, key, details):
        model = details['model_uri'].split('/')[-2]
        prices = PRICES_RUB_PER_1000[model]
        cost = (details['prompt_tokens'] * prices['input'] + details['completion_tokens'] * prices['output']) / 1000
        self.connection.execute('INSERT INTO usage_events\n            (namespace,cache_key,model_uri,prompt_tokens,completion_tokens,total_tokens,\n             reasoning_tokens,finish_reason,max_tokens,cost_rub,timestamp_utc)\n            VALUES (?,?,?,?,?,?,?,?,?,?,?)', (self.namespace, key, details['model_uri'], details['prompt_tokens'], details['completion_tokens'], details['total_tokens'], details.get('reasoning_tokens'), details.get('finish_reason'), details.get('max_tokens'), cost, datetime.now(timezone.utc).isoformat()))

    def total_spend(self):
        with self.lock:
            return float(self.connection.execute('SELECT COALESCE(SUM(cost_rub),0) FROM usage_events').fetchone()[0])

    def usage_frame(self):
        with self.lock:
            return pd.read_sql_query('SELECT * FROM usage_events WHERE namespace=? ORDER BY event_id', self.connection, params=[self.namespace])

    def close(self):
        self.connection.close()
_original_create = openai_client.chat.completions.create

def create_with_usage_observation(**kwargs):
    if isinstance(llm_cache, PersistentSubsetCache):
        used = llm_cache.total_spend()
        if used >= BUDGET_LIMIT_RUB - BUDGET_SAFETY_BUFFER_RUB:
            raise RuntimeError(f'Local recorded spend is {used:.2f} RUB; budget safety threshold reached. Cached unfinished work is retained.')
    response = _original_create(**kwargs)
    usage = _extract_openai_usage(getattr(response, 'usage', None))
    usage_object = getattr(response, 'usage', None)
    completion_details = getattr(usage_object, 'completion_tokens_details', None)
    choices = getattr(response, 'choices', None) or []
    _request_details.latest = {'model_uri': kwargs['model'], **usage, 'reasoning_tokens': getattr(completion_details, 'reasoning_tokens', None), 'finish_reason': getattr(choices[0], 'finish_reason', None) if choices else None, 'max_tokens': kwargs.get('max_tokens')}
    return response
openai_client.chat.completions.create = create_with_usage_observation
_original_call_llm = call_llm

def call_llm_with_model_limits(provider, model_uri, system_text, user_text, temperature, max_tokens):
    if isinstance(llm_cache, PersistentSubsetCache) and llm_cache.total_spend() >= BUDGET_LIMIT_RUB - BUDGET_SAFETY_BUFFER_RUB:
        raise RuntimeError('Recorded-spend safety threshold reached. Resume after reviewing usage and the budget setting.')
    model = model_uri.split('/')[-2]
    effective_limit = MODEL_MAX_OUTPUT_TOKENS.get(model, max_tokens)
    _request_details.latest = None
    return _original_call_llm(provider, model_uri, system_text, user_text, temperature, effective_limit)
call_llm = call_llm_with_model_limits


In [ ]:
SOURCE_SHA256 = '3802b4a8480d8f884281972f6a3198882006e91385b564e54a0c684d18639c24'
SYSTEM_ID = 'baseline_gpt120'
PREDICTION_COLUMN = 'baseline_selected_option'
ORIGINAL_PIPELINE = run_baseline_pipeline

def ORIGINAL_SUMMARIES(frame):
    return [('token_summary_df', build_token_summary(frame))]
RUN_CONFIGURATION = {'system': SYSTEM_ID, 'source_sha256': SOURCE_SHA256, 'model_uri': BASELINE_MODEL_URI, 'role_name': BASELINE_ROLE_NAME, 'role_description': BASELINE_ROLE_DESCRIPTION, 'temperature': TEMPERATURE_PROTOCOL, 'model_output_limits': MODEL_MAX_OUTPUT_TOKENS, 'reasoning_mode': 'provider_default'}


In [ ]:
def atomic_json(path, content):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(content, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def atomic_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, sep=';', index=False, encoding='utf-8-sig')
    temporary.replace(path)

def wilson_interval(correct, n):
    z = 1.959963984540054
    proportion = correct / n
    denominator = 1 + z * z / n
    center = (proportion + z * z / (2 * n)) / denominator
    radius = z * math.sqrt(proportion * (1 - proportion) / n + z * z / (4 * n * n)) / denominator
    return [center - radius, center + radius]

def run_subset_sequence():
    global llm_cache, df, result_df
    output_directory = RUNTIME_DIR / SYSTEM_ID
    output_directory.mkdir(parents=True, exist_ok=True)
    completed_metrics = []
    for subset_id in SUBSET_ORDER:
        specification = SUBSETS[subset_id]
        df = load_subset(subset_id)
        signature = hashlib.sha256(json.dumps({'configuration': RUN_CONFIGURATION, 'input_sha256': specification['sha256']}, sort_keys=True).encode()).hexdigest()
        namespace = f'{SYSTEM_ID}/{subset_id}/{signature}'
        llm_cache = PersistentSubsetCache(LEDGER_PATH, namespace)
        output_csv = output_directory / f'{subset_id}.csv'
        metrics_json = output_directory / f'{subset_id}.metrics.json'
        checkpoint_csv = output_directory / f'{subset_id}.partial.csv'
        checkpoint_json = output_directory / f'{subset_id}.partial.json'
        print(f"\n{subset_id}: n={len(df)}; split={specification['split']}")
        if output_csv.exists() and (not metrics_json.exists()):
            if not (checkpoint_csv.exists() and checkpoint_json.exists()):
                llm_cache.close()
                raise RuntimeError('Incomplete final output with no recovery checkpoint; retain the files for inspection.')
            print('Resuming interrupted finalization from the checkpoint.')
        elif output_csv.exists() or metrics_json.exists():
            if not output_csv.exists():
                llm_cache.close()
                raise RuntimeError('Final metrics exist but their result CSV is missing; retain the files for inspection.')
            saved = json.loads(metrics_json.read_text(encoding='utf-8'))
            if saved['configuration_signature'] != signature or saved['output_sha256'] != sha256_file(output_csv):
                llm_cache.close()
                raise ValueError('Existing result has a different configuration/checksum. Choose a new RUN_TAG for a new experiment.')
            if SKIP_COMPLETED:
                llm_cache.clear()
                llm_cache.close()
                completed_metrics.append(saved)
                print('Already complete:', output_csv)
                continue
            llm_cache.close()
            raise ValueError('Use a new RUN_TAG to repeat a completed experiment without overwriting it.')
        accumulated = pd.DataFrame()
        if checkpoint_csv.exists():
            if not checkpoint_json.exists():
                llm_cache.close()
                raise ValueError('Checkpoint metadata is missing.')
            checkpoint = json.loads(checkpoint_json.read_text(encoding='utf-8'))
            if checkpoint['configuration_signature'] != signature:
                llm_cache.close()
                raise ValueError('Checkpoint configuration differs; use a new RUN_TAG.')
            accumulated = pd.read_csv(checkpoint_csv, sep=';', encoding='utf-8-sig', dtype={'case_id': str})
            if accumulated['case_id'].duplicated().any() or not set(accumulated['case_id']).issubset(set(df['case_id'])):
                llm_cache.close()
                raise ValueError('Invalid checkpoint case IDs.')
        seen = set(accumulated['case_id']) if len(accumulated) else set()
        remaining = df.loc[~df['case_id'].isin(seen)].copy()
        atomic_json(checkpoint_json, {'configuration_signature': signature, 'completed_cases': len(accumulated)})
        try:
            for start in range(0, len(remaining), CHECKPOINT_BATCH_SIZE):
                batch = remaining.iloc[start:start + CHECKPOINT_BATCH_SIZE].copy()
                batch_result = ORIGINAL_PIPELINE(batch)
                accumulated = pd.concat([accumulated, batch_result], ignore_index=True, sort=False)
                atomic_csv(accumulated, checkpoint_csv)
                atomic_json(checkpoint_json, {'configuration_signature': signature, 'completed_cases': len(accumulated)})
                print(f'Checkpoint {len(accumulated)}/{len(df)}; recorded combined spend {llm_cache.total_spend():.2f} RUB')
            if set(accumulated['case_id']) != set(df['case_id']):
                raise ValueError('Final results do not cover the intended sample.')
            result_df = df[['case_id']].merge(accumulated, on='case_id', how='left', validate='one_to_one')
            result_df[PREDICTION_COLUMN] = result_df[PREDICTION_COLUMN].fillna('')
            result_part = result_df.copy()
            result_part['table_source'] = 'result_df'
            parts = [result_part]
            for source_name, frame in ORIGINAL_SUMMARIES(result_df):
                frame = frame.copy()
                frame['table_source'] = source_name
                parts.append(frame)
            combined_output = pd.concat(parts, ignore_index=True, sort=False)
            combined_output['subset_id'] = subset_id
            atomic_csv(combined_output, output_csv)
            usage = llm_cache.usage_frame()
            atomic_csv(usage, output_directory / f'{subset_id}.usage.csv')
            prediction = result_df[PREDICTION_COLUMN].fillna('').astype(str).str.strip().str.upper()
            valid = prediction.isin(ANSWER_LABELS)
            correct = int((valid & prediction.eq(result_df['correct_option_letter'])).sum())
            n = len(result_df)
            metrics = {'system_id': SYSTEM_ID, 'subset_id': subset_id, 'n': n, 'correct': correct, 'invalid_answers': int((~valid).sum()), 'accuracy': correct / n, 'accuracy_wilson_95ci': wilson_interval(correct, n), 'split': specification['split'], 'source_url': specification['source_url'], 'configuration_signature': signature, 'configuration': RUN_CONFIGURATION, 'input_sha256': specification['sha256'], 'output_sha256': sha256_file(output_csv), 'billed_prompt_tokens': int(usage['prompt_tokens'].sum()) if len(usage) else 0, 'billed_completion_tokens': int(usage['completion_tokens'].sum()) if len(usage) else 0, 'recorded_cost_rub': float(usage['cost_rub'].sum()) if len(usage) else 0.0, 'length_limited_calls': int(usage['finish_reason'].eq('length').sum()) if len(usage) else 0, 'price_source': PRICE_SOURCE, 'cache_policy': 'persist during the subset; clear responses after successful final write'}
            atomic_json(metrics_json, metrics)
            llm_cache.clear()
            completed_metrics.append(metrics)
            print(f'Saved {output_csv}; accuracy={correct}/{n} ({correct / n:.3%}); cache cleared.')
        finally:
            llm_cache.close()
    summary = pd.DataFrame([{key: record[key] for key in ['system_id', 'subset_id', 'n', 'correct', 'invalid_answers', 'accuracy', 'recorded_cost_rub', 'length_limited_calls']} for record in completed_metrics])
    atomic_csv(summary, output_directory / 'all_subsets_summary.csv')
    return summary
summary_df = run_subset_sequence()
summary_df
